# Neoplasms (Cancer) Drilldown Analysis

This notebook performs a drilldown analysis into neoplasms to identify which specific cancers and risk factors are driving the gender gap in Life Expectancy and HALE.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from utils import (
    decorate, configure_plot_style, AIBM_COLORS, write_html_table
)

configure_plot_style()

## Data Loading

We have two data files from IHME GBD 2023 for the United States, separated by sex. These files contain death rates attributable to specific risk factors.

In [3]:
location = 'usa' # 'usa', 'iceland', or 'oecd'
male_file = f'../data/ihme_cancer_drilldown_{location}_male.csv'
female_file = f'../data/ihme_cancer_drilldown_{location}_female.csv'

male_df = pd.read_csv(male_file)
female_df = pd.read_csv(female_file)

# Replace NaN with 0 in the Value column before analysis
male_df['Value'] = male_df['Value'].fillna(0)
female_df['Value'] = female_df['Value'].fillna(0)

print(f"Male data shape: {male_df.shape}")
print(f"Female data shape: {female_df.shape}")

Male data shape: (307, 11)
Female data shape: (307, 11)


### Basic Inventory

Let's look at the structure of the data.

In [4]:
male_df.head()

,Population,Location,Year,Age,Sex,Cause of death or injury,Risk factor,Measure,Value,Lower bound,Upper bound
0,All Population,United States of America,2023.0,All ages,Male,Lip and oral cavity cancer,Metabolic risks,Deaths,0.0,NaN,NaN
1,All Population,United States of America,2023.0,All ages,Male,Lip and oral cavity cancer,Metabolic risks,Percent of total deaths,0.0,NaN,NaN
2,All Population,United States of America,2023.0,All ages,Male,Lip and oral cavity cancer,Metabolic risks,"Deaths per 100,000",0.0,NaN,NaN
3,All Population,United States of America,2023.0,All ages,Male,Lip and oral cavity cancer,Environmental/occupational risks,Deaths,0.0,NaN,NaN
4,All Population,United States of America,2023.0,All ages,Male,Lip and oral cavity cancer,Environmental/occupational risks,Percent of total deaths,0.0,NaN,NaN


### Column Analysis

In [5]:
print("Columns in the dataset:")
for col in male_df.columns:
    print(f"- {col}")

Columns in the dataset:
- Population
- Location
- Year
- Age
- Sex
- Cause of death or injury
- Risk factor
- Measure
- Value
- Lower bound
- Upper bound


### Unique Values

Let's see what categories we are working with.

In [6]:
print("Unique Causes:")
male_df['Cause of death or injury'].unique()

Unique Causes:


array(['Lip and oral cavity cancer', 'Nasopharynx cancer',
       'Other pharynx cancer', 'Esophageal cancer', 'Stomach cancer',
       'Colon and rectum cancer', 'Liver cancer',
       'Gallbladder and biliary tract cancer', 'Pancreatic cancer',
       'Larynx cancer', 'Tracheal, bronchus, and lung cancer',
       'Malignant skin melanoma', 'Non-melanoma skin cancer',
       'Soft tissue and other extraosseous sarcomas',
       'Malignant neoplasm of bone and articular cartilage',
       'Breast cancer', 'Cervical cancer', 'Uterine cancer',
       'Ovarian cancer', 'Prostate cancer', 'Testicular cancer',
       'Kidney cancer', 'Bladder cancer',
       'Brain and central nervous system cancer', 'Eye cancer',
       'Neuroblastoma and other peripheral nervous cell tumors',
       'Thyroid cancer', 'Mesothelioma', 'Hodgkin lymphoma',
       'Non-Hodgkin lymphoma', 'Multiple myeloma', 'Leukemia',
       'Other malignant neoplasms', 'Other neoplasms', nan], dtype=object)

In [7]:
print("Unique Risk Factors:")
male_df['Risk factor'].unique()

Unique Risk Factors:


array(['Metabolic risks', 'Environmental/occupational risks',
       'Behavioral risks', nan], dtype=object)

In [8]:
print("Unique Measures:")
male_df['Measure'].unique()

Unique Measures:


array(['Deaths', 'Percent of total deaths', 'Deaths per 100,000', nan],
      dtype=object)

## Missing Value Analysis

Let's check for missing values in both datasets.

In [9]:
def check_missing(df, name):
    print(f"\nMissing values in {name}:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("No missing values found.")
    else:
        display(missing[missing > 0])

check_missing(male_df, "Male Data")
check_missing(female_df, "Female Data")


Missing values in Male Data:


Location                      1
Year                          1
Age                           1
Sex                           1
Cause of death or injury      1
Risk factor                   1
Measure                       1
Lower bound                 243
Upper bound                 243
dtype: int64


Missing values in Female Data:


Location                      1
Year                          1
Age                           1
Sex                           1
Cause of death or injury      1
Risk factor                   1
Measure                       1
Lower bound                 235
Upper bound                 235
dtype: int64

### Data Availability by Cause and Risk Factor

Not all cancers have data for all three risk categories (Behavioral, Metabolic, Environmental). Let's see which ones do.

In [10]:
def check_data_availability(df, name):
    print(f"\nData availability for {name} (Measure='Deaths per 100,000', Value > 0):")
    deaths_rate = df[df['Measure'] == 'Deaths per 100,000']
    availability = deaths_rate.groupby(['Cause of death or injury', 'Risk factor'])['Value'].apply(lambda x: (x > 0).sum()).unstack().fillna(0)
    display(availability)
    return availability

male_availability = check_data_availability(male_df, "Male")
female_availability = check_data_availability(female_df, "Female")


Data availability for Male (Measure='Deaths per 100,000', Value > 0):


Risk factor,Behavioral risks,Environmental/occupational risks,Metabolic risks
Cause of death or injury,,,
Bladder cancer,1,0,1
Brain and central nervous system cancer,0,0,0
Breast cancer,1,0,0
Cervical cancer,0,0,0
Colon and rectum cancer,1,0,1
Esophageal cancer,1,0,0
Eye cancer,0,0,0
Gallbladder and biliary tract cancer,0,0,1
Hodgkin lymphoma,0,0,0



Data availability for Female (Measure='Deaths per 100,000', Value > 0):


Risk factor,Behavioral risks,Environmental/occupational risks,Metabolic risks
Cause of death or injury,,,
Bladder cancer,1,0,1
Brain and central nervous system cancer,0,0,0
Breast cancer,1,0,1
Cervical cancer,1,0,0
Colon and rectum cancer,1,0,1
Esophageal cancer,1,0,0
Eye cancer,0,0,0
Gallbladder and biliary tract cancer,0,0,1
Hodgkin lymphoma,0,0,0


## Risk Factor Fractions

Before aggregating the data, let's see what fraction of the attributable death rate for each cancer is assigned to each risk factor category.

In [11]:
def get_risk_fractions(df):
    # Filter for death rates
    df_filtered = df[df['Measure'] == 'Deaths per 100,000'].copy()
    
    # Pivot to get risk factors as columns
    pivot = df_filtered.pivot_table(
        index='Cause of death or injury', 
        columns='Risk factor', 
        values='Value', 
        fill_value=0
    )
    
    # Calculate total attributable rate for each cancer
    row_totals = pivot.sum(axis=1)
    
    # Calculate fractions (avoiding division by zero)
    fractions = pivot.divide(row_totals, axis=0).fillna(0)
    
    return fractions

print("Risk Factor Fractions (Male):")
male_fractions = get_risk_fractions(male_df)
male_fractions.head(10)

Risk Factor Fractions (Male):


Risk factor,Behavioral risks,Environmental/occupational risks,Metabolic risks
Cause of death or injury,,,
Bladder cancer,0.659808,0.000000,0.340192
Brain and central nervous system cancer,0.000000,0.000000,0.000000
Breast cancer,1.000000,0.000000,0.000000
Cervical cancer,0.000000,0.000000,0.000000
Colon and rectum cancer,0.665197,0.000000,0.334803
Esophageal cancer,1.000000,0.000000,0.000000
Eye cancer,0.000000,0.000000,0.000000
Gallbladder and biliary tract cancer,0.000000,0.000000,1.000000
Hodgkin lymphoma,0.000000,0.000000,0.000000


In [12]:
print("Risk Factor Fractions (Female):")
female_fractions = get_risk_fractions(female_df)
female_fractions.head(10)

Risk Factor Fractions (Female):


Risk factor,Behavioral risks,Environmental/occupational risks,Metabolic risks
Cause of death or injury,,,
Bladder cancer,0.624216,0.000000,0.375784
Brain and central nervous system cancer,0.000000,0.000000,0.000000
Breast cancer,0.574813,0.000000,0.425187
Cervical cancer,1.000000,0.000000,0.000000
Colon and rectum cancer,0.683348,0.000000,0.316652
Esophageal cancer,1.000000,0.000000,0.000000
Eye cancer,0.000000,0.000000,0.000000
Gallbladder and biliary tract cancer,0.000000,0.000000,1.000000
Hodgkin lymphoma,0.000000,0.000000,0.000000


## Data Processing

We focus on the "Deaths per 100,000" measure. To find the total death rate attributable to the risk factors included in this dataset, we sum across the risk categories (Behavioral, Metabolic, Environmental/occupational) for each cancer type.

In [13]:
def process_cancer_data(df):
    # Filter for death rates
    df_filtered = df[df['Measure'] == 'Deaths per 100,000'].copy()
    
    # Sum across risk factors to get total attributable death rate per cancer
    df_agg = df_filtered.groupby('Cause of death or injury')['Value'].sum().reset_index()
    
    return df_agg

male_rates = process_cancer_data(male_df)
female_rates = process_cancer_data(female_df)

# Merge male and female rates
merged_rates = pd.merge(
    male_rates, female_rates, 
    on='Cause of death or injury', 
    suffixes=('_Male', '_Female'),
    how='outer'
)

# Fill any remaining NaNs after the outer merge (for sex-specific cancers)
merged_rates = merged_rates.fillna(0)

# Rename columns for clarity
merged_rates.columns = ['Neoplasm', 'Male Rate', 'Female Rate']

## Computing the Gender Gap

We calculate the difference between male and female death rates. A positive difference indicates a higher death rate for males.

In [14]:
# Calculate the difference
merged_rates['Difference'] = merged_rates['Male Rate'] - merged_rates['Female Rate']

# Sort by the absolute value of the difference to find the biggest drivers
merged_rates['abs_diff'] = merged_rates['Difference'].abs()
gap_table = merged_rates.sort_values(by='abs_diff', ascending=False).drop(columns=['abs_diff'])

# Display the top contributors
gap_table.head(10)

,Neoplasm,Male Rate,Female Rate,Difference
32,"Tracheal, bronchus, and lung cancer",52.950447,34.571181,18.379266
2,Breast cancer,0.074071,12.522016,-12.447946
5,Esophageal cancer,7.093973,1.406449,5.687524
13,Liver cancer,10.521352,4.842568,5.678784
3,Cervical cancer,0.000000,4.193822,-4.193822
33,Uterine cancer,0.000000,3.355938,-3.355938
0,Bladder cancer,3.378072,1.053684,2.324388
4,Colon and rectum cancer,18.192874,16.123975,2.068899
25,Ovarian cancer,0.000000,1.832406,-1.832406
11,Leukemia,3.933465,2.228891,1.704573


## Exporting Results

We export the full gap table to an HTML file for inclusion in the project's technical report.

In [15]:
# Write to HTML
write_html_table(gap_table, f'jb/tables/neoplasms_gap_{location}.html')
print(f"Table written to jb/tables/neoplasms_gap_{location}.html")

Table written to jb/tables/neoplasms_gap_usa.html


## Risk Factor Analysis

While the table above shows the total attributable gap, we can also look at which risk factor categories are the primary drivers.

In [16]:
def get_risk_breakdown(df, sex):
    df_filtered = df[df['Measure'] == 'Deaths per 100,000'].copy()
    breakdown = df_filtered.pivot_table(
        index='Cause of death or injury', 
        columns='Risk factor', 
        values='Value', 
        fill_value=0
    )
    breakdown.columns = [f"{col}_{sex}" for col in breakdown.columns]
    return breakdown

male_risk = get_risk_breakdown(male_df, 'Male')
female_risk = get_risk_breakdown(female_df, 'Female')

risk_merged = pd.concat([male_risk, female_risk], axis=1).fillna(0)

# Calculate gap by risk factor
for risk in ['Behavioral risks', 'Environmental/occupational risks', 'Metabolic risks']:
    risk_merged[f'Gap_{risk}'] = risk_merged[f'{risk}_Male'] - risk_merged[f'{risk}_Female']

# Select and sort gap columns
gap_cols = [col for col in risk_merged.columns if col.startswith('Gap_')]
risk_gap_summary = risk_merged[gap_cols].copy()
risk_gap_summary['Total Gap'] = risk_gap_summary.sum(axis=1)

# Add Neoplasm column and sort
risk_gap_summary.index.name = 'Neoplasm'
risk_gap_summary = risk_gap_summary.reset_index()
risk_gap_summary = risk_gap_summary.sort_values(by='Total Gap', ascending=False, key=abs)

# Export risk gap summary to HTML
write_html_table(risk_gap_summary, f'jb/tables/neoplasms_risk_gap_{location}.html')
print(f"Risk gap table written to jb/tables/neoplasms_risk_gap_{location}.html")

risk_gap_summary

Risk gap table written to jb/tables/neoplasms_risk_gap_usa.html


,Neoplasm,Gap_Behavioral risks,Gap_Environmental/occupational risks,Gap_Metabolic risks,Total Gap
32,"Tracheal, bronchus, and lung cancer",6.765224,11.111537,0.502506,18.379266
2,Breast cancer,-7.123751,0.000000,-5.324195,-12.447946
5,Esophageal cancer,5.687524,0.000000,0.000000,5.687524
13,Liver cancer,4.539114,0.000000,1.139670,5.678784
3,Cervical cancer,-4.193822,0.000000,0.000000,-4.193822
33,Uterine cancer,0.000000,0.000000,-3.355938,-3.355938
0,Bladder cancer,1.571151,0.000000,0.753237,2.324388
4,Colon and rectum cancer,1.083562,0.000000,0.985337,2.068899
25,Ovarian cancer,0.000000,-0.462673,-1.369733,-1.832406
11,Leukemia,1.294905,0.001557,0.408111,1.704573


In [17]:
# compute the sum of Total Gap for all positive gaps and all negative gaps 
pos_sum = risk_gap_summary.loc[risk_gap_summary['Total Gap'] > 0, 'Total Gap'].sum()
neg_sum = risk_gap_summary.loc[risk_gap_summary['Total Gap'] < 0, 'Total Gap'].sum()

print(f"Sum of positive gaps: {pos_sum:.2f}")
print(f"Sum of negative gaps: {neg_sum:.2f}")

Sum of positive gaps: 45.70
Sum of negative gaps: -21.93


In [18]:
# display the index of the rows where Total Gap is exactly zero 
risk_gap_summary[risk_gap_summary['Total Gap'] == 0].index

Index([1, 6, 8, 14, 15, 19, 21, 22, 23, 28, 30], dtype='int64')